# SMM Route Optimizer — Rapport de Projet

**Sujet :** Optimisation de routes inspirée de la logistique, sur un graphe généré procéduralement  
**Stack :** Next.js (TypeScript) · Python FastAPI · Dijkstra + TSP (plus proche voisin + 2-opt)

---

## 1. Présentation du Projet

L'objectif est de trouver les routes optimales pour un ou plusieurs véhicules visitant un ensemble de villes sur un graphe pondéré, en minimisant l'un des trois critères : **temps total**, **distance totale** ou **consommation de carburant totale**.

L'application est structurée en trois couches :

| Couche | Technologie | Rôle |
|--------|------------|------|
| Frontend | Next.js 15 / React 19 | Éditeur de carte, interface objectifs, lecture d'animation |
| Solveur JS | TypeScript (navigateur) | Solveur par défaut, s'exécute entièrement côté client |
| Solveur Python | FastAPI + uvicorn | Solveur backend optionnel, détecté automatiquement via `/health` |

Les deux solveurs implémentent le même algorithme. Le solveur Python est activé automatiquement lorsque le serveur FastAPI tourne sur le port 8000.

---

## 2. Structure du Graphe

La carte est modélisée comme un **multigraphe pondéré non orienté** $G = (V, E)$.

### Nœuds

Il existe deux types de nœuds :

- **Node** (`type = 'node'`) : une ville ou un lieu principal — le seul type sélectionnable comme destination.
- **Subnode** (`type = 'subnode'`) : un carrefour routier, utilisé pour donner aux arêtes une géométrie courbe ou indirecte. Invisible pour l'utilisateur en tant que destination.

### Arêtes

Chaque arête relie deux nœuds (ou subnodes) et est composée d'un ou plusieurs **segments**. Chaque segment contient :

| Champ | Type | Description |
|-------|------|-------------|
| `distance` | km | Longueur physique |
| `speed` | km/h | Limitation de vitesse |
| `traffic` | [0, 1] | Facteur de charge du trafic |

### Distance totale d'une arête

$$d_e = \sum_{s \in e} d_s$$

### Vitesse effective sur un segment

Le trafic réduit la vitesse effective via un facteur d'efficacité plafonné :

$$\text{eff}_s = \max(0{,}05,\ 1 - 0{,}6 \cdot t_s)$$

$$v_s^{\text{eff}} = \min(v_s,\ v^{\text{max}}) \cdot \text{eff}_s$$

où $t_s \in [0,1]$ est la charge de trafic et $v^{\text{max}}$ est la vitesse maximale du véhicule.

### Temps de trajet sur un segment (en minutes)

$$T_s = \frac{d_s}{v_s^{\text{eff}}} \times 60$$

### Consommation de carburant sur un segment (en litres)

Un modèle linéaire dépendant du trafic :

$$F_s = d_s \cdot (0{,}07 + 0{,}05 \cdot t_s)$$

À trafic nul, cela donne **7 L/100 km** ; à trafic maximal, cela monte à **12 L/100 km**.

---

## 3. Pipeline Algorithmique

Le traitement se déroule en six étapes séquentielles :

1. **Construction du graphe** — pour chaque vitesse maximale de véhicule distincte, on construit une liste d'adjacence où chaque arête stocke son coût en temps, distance et carburant.
2. **Dijkstra tous-pairs** — pour chaque ville à visiter, on lance Dijkstra depuis cette ville comme source. On obtient une matrice de coûts entre toutes les paires de villes.
3. **Affectation des villes** — les villes et les cargaisons sont réparties entre les instances de véhicules selon les règles de capacité.
4. **Résolution TSP** — pour chaque véhicule, on construit un tour par plus proche voisin puis on l'améliore avec l'algorithme 2-opt.
5. **Développement du chemin complet** — le tour TSP (suite de villes) est développé en chemin détaillé passant par tous les nœuds et arêtes intermédiaires.
6. **Métriques et contraintes** — on calcule la distance, le temps et le carburant du chemin final, puis on vérifie les contraintes éventuelles (temps max, distance max, capacité).

La sortie contient les métriques finales et une liste de *frames* pour l'animation de la simulation.

---

## 4. Dijkstra — Plus Court Chemin

L'algorithme de Dijkstra trouve le chemin de coût minimal depuis un nœud source vers tous les autres nœuds dans un graphe à poids positifs.

**Complexité :** $O((V + E) \log V)$ avec un tas binaire.

### Condition de relaxation

Pour chaque voisin $v$ du nœud courant $u$ :

$$\text{coût}[v] > \text{coût}[u] + w(u, v) \implies \text{coût}[v] \leftarrow \text{coût}[u] + w(u, v)$$

Le poids $w(u,v)$ est l'un des trois critères $T_e$, $d_e$ ou $F_e$ selon l'objectif d'optimisation.

L'implémentation effectue **un passage Dijkstra par ville** (utilisée comme source), produisant une matrice de coût tous-pairs sur les villes à visiter. Cette matrice est réutilisée pour le TSP.

---

## 5. TSP — Plus Proche Voisin + 2-opt

Le Problème du Voyageur de Commerce (TSP) consiste à trouver le tour de coût minimal visitant chaque ville exactement une fois, étant données les villes et leurs coûts par paires.

Le TSP est NP-difficile. On utilise une heuristique en deux phases :

### Phase 1 — Construction par plus proche voisin

En partant de la ville de départ (ou de la ville dont la distance moyenne aux autres est minimale), on choisit goulûment la ville non visitée la plus proche à chaque étape.

$$\text{suivant} = \arg\min_{c \notin \text{visités}} \text{coût}(\text{courant}, c)$$

**Complexité :** $O(n^2)$

### Phase 2 — Amélioration 2-opt

On inverse itérativement des sous-séquences de la route pour réduire le coût total. Un échange des arêtes $(i, i+1)$ et $(j, j+1)$ est accepté si :

$$\text{coût}(r_i, r_j) + \text{coût}(r_{i+1}, r_{j+1}) < \text{coût}(r_i, r_{i+1}) + \text{coût}(r_j, r_{j+1})$$

La boucle s'exécute au maximum 20 itérations ou jusqu'à ce qu'aucune amélioration ne soit trouvée.

**Complexité par itération :** $O(n^2)$

---

## 6. Référence du Code — Fonctions Clés de `scripts/solver.py`

Les cellules suivantes reproduisent les algorithmes principaux du solveur Python pour les rendre exécutables et lisibles directement dans ce notebook.

### 6.1 Construction du graphe d'adjacence

Cette fonction transforme la liste des arêtes en dictionnaire d'adjacence. Pour chaque segment d'une arête, elle calcule le temps de trajet effectif (en tenant compte du trafic et de la vitesse max du véhicule), la distance et la consommation de carburant. Chaque arête est ajoutée dans les deux sens (graphe non orienté).

In [ ]:
import heapq, math
from dataclasses import dataclass
from typing import Any

@dataclass
class GEdge:
    id: str
    frm: str
    to: str
    dist: float   # km
    time: float   # minutes
    fuel: float   # litres

@dataclass
class Segment:
    distance: float   # km
    speed: float      # km/h
    traffic: float    # 0-1

@dataclass
class Edge:
    id: str
    nodeA: str
    nodeB: str
    segments: list
    totalDistance: float = 0.0

def build_adj(edges: list, nodes: list[str], vehicle_speed_max: float = math.inf) -> dict[str, list]:
    adj: dict[str, list] = {n: [] for n in nodes}
    for e in edges:
        dist = time_ = fuel = 0.0
        for s in e.segments:
            dist  += s.distance
            eff    = max(0.05, 1 - s.traffic * 0.6)          # réduction due au trafic
            v_eff  = min(s.speed, vehicle_speed_max) * eff    # vitesse effective
            time_ += (s.distance / v_eff) * 60               # minutes
            fuel  += s.distance * (0.07 + s.traffic * 0.05)  # litres
        for frm, to in [(e.nodeA, e.nodeB), (e.nodeB, e.nodeA)]:
            adj.setdefault(frm, []).append(GEdge(e.id, frm, to, dist, time_, fuel))
    return adj

print("build_adj défini")

### 6.2 Dijkstra

`dijkstra()` explore le graphe depuis un nœud source avec un tas min (heap), et retourne le coût minimal vers chaque nœud ainsi que les pointeurs permettant de reconstruire le chemin. `get_path()` remonte ces pointeurs pour produire la liste ordonnée des nœuds et des arêtes entre deux villes.

In [ ]:
def dijkstra(adj: dict, source: str, key: str):
    cost      = {n: math.inf for n in adj}
    prev      = {n: None     for n in adj}
    prev_edge = {n: None     for n in adj}
    cost[source] = 0.0
    pq: list = [(0.0, source)]
    vis: set = set()
    while pq:
        c, u = heapq.heappop(pq)
        if u in vis:
            continue
        vis.add(u)
        for e in adj.get(u, []):
            nc = c + getattr(e, key)   # key = "time", "dist" ou "fuel"
            if nc < cost.get(e.to, math.inf):
                cost[e.to]      = nc
                prev[e.to]      = u
                prev_edge[e.to] = e.id
                heapq.heappush(pq, (nc, e.to))
    return cost, prev, prev_edge

def get_path(cost, prev, prev_edge, src, tgt):
    if math.isinf(cost.get(tgt, math.inf)):
        return {"nodes": [], "edges": [], "cost": math.inf}
    nodes, edges = [], []
    cur = tgt
    while cur is not None and cur != src:
        nodes.insert(0, cur)
        e = prev_edge.get(cur)
        if e:
            edges.insert(0, e)
        cur = prev.get(cur)
    nodes.insert(0, src)
    return {"nodes": nodes, "edges": edges, "cost": cost[tgt]}

print("dijkstra défini")

### 6.3 TSP — Plus proche voisin + 2-opt

`solve_tsp()` prend la liste des villes à visiter, une ville de départ optionnelle et une ville d'arrivée optionnelle. Elle construit d'abord un tour par greedy (plus proche voisin), puis tente d'améliorer ce tour en testant toutes les inversions de sous-séquences possibles (2-opt), jusqu'à 20 itérations.

In [ ]:
def solve_tsp(normals: list, start, end, costs: dict) -> list:
    def get_c(a, b):
        return min(costs.get(f"{a}|{b}", math.inf),
                   costs.get(f"{b}|{a}", math.inf))

    if not normals:
        return [c for c in [start, end] if c is not None]

    to_visit = list(normals)
    route = []

    # Phase 1 : construction par plus proche voisin
    if start:
        route.append(start)
        current = start
    else:
        # Pas de départ fixé : on choisit la ville la plus "centrale"
        best = min(to_visit, key=lambda c: sum(get_c(c, o) for o in to_visit if o != c) / max(len(to_visit)-1, 1))
        to_visit.remove(best)
        route.append(best)
        current = best

    while to_visit:
        nn = min(to_visit, key=lambda c: get_c(current, c))
        to_visit.remove(nn)
        route.append(nn)
        current = nn

    if end and route[-1] != end:
        route.append(end)

    # Phase 2 : amélioration 2-opt (on ne touche pas aux villes de départ/arrivée fixées)
    sf = 1 if start else 0
    ef = 1 if end   else 0
    def tour_cost(r): return sum(get_c(r[i], r[i+1]) for i in range(len(r)-1))

    for _ in range(20):
        improved = False
        for i in range(sf, len(route) - 1 - ef):
            for j in range(i + 1, len(route) - ef):
                nr = route[:i+1] + route[i+1:j+1][::-1] + route[j+1:]
                if tour_cost(nr) < tour_cost(route) - 0.001:
                    route = nr
                    improved = True
        if not improved:
            break

    return route

print("solve_tsp défini")

---

## 7. Exemple — Démonstration à Petite Échelle

On construit un graphe de 5 villes reliées par 6 routes, chacune avec une distance, une vitesse limite et un niveau de trafic. On exécute ensuite les trois étapes : construction du graphe, calcul des plus courts chemins, puis résolution d'un tour TSP.

```
    A ---10--- B
    |  \       |
    4    15    7
    |      \   |
    C ---9-- D-3-E
```

**Cellule 1** — Construction du graphe et affichage de la liste d'adjacence avec les coûts calculés.

In [ ]:
nodes = ["A", "B", "C", "D", "E"]

raw_edges = [
    # (id, nœud A, nœud B, distance km, vitesse km/h, trafic)
    ("e1", "A", "B", 10.0, 90,  0.1),
    ("e2", "A", "C",  4.0, 50,  0.3),
    ("e3", "A", "D", 15.0, 110, 0.0),
    ("e4", "B", "D",  7.0, 90,  0.5),
    ("e5", "C", "D",  9.0, 70,  0.2),
    ("e6", "D", "E",  3.0, 50,  0.0),
]

edges = [Edge(eid, a, b, [Segment(d, spd, traf)], d) for eid, a, b, d, spd, traf in raw_edges]
adj = build_adj(edges, nodes)

print("Graphe construit — liste d'adjacence :")
for node, neighbors in adj.items():
    for e in neighbors:
        print(f"  {e.frm} -> {e.to}  dist={e.dist:.1f} km  temps={e.time:.2f} min  carburant={e.fuel:.3f} L")

**Cellule 2** — Calcul des plus courts chemins entre toutes les paires de villes (critère : temps). On affiche la matrice des coûts.

In [ ]:
cities = ["A", "B", "C", "D", "E"]
pair_paths = {}
pair_costs = {}

for src in cities:
    c, prev, prev_edge = dijkstra(adj, src, "time")
    for tgt in cities:
        if tgt == src:
            continue
        p = get_path(c, prev, prev_edge, src, tgt)
        pair_paths[f"{src}|{tgt}"] = p
        pair_costs[f"{src}|{tgt}"] = p["cost"]

print("Plus courts chemins tous-pairs (temps en minutes) :")
print(f"{'':4}", end="")
for tgt in cities:
    print(f"{tgt:>8}", end="")
print()
for src in cities:
    print(f"{src:4}", end="")
    for tgt in cities:
        if src == tgt:
            print(f"{'—':>8}", end="")
        else:
            v = pair_costs.get(f"{src}|{tgt}", math.inf)
            print(f"{v:>8.2f}", end="")
    print()

**Cellule 3** — Résolution TSP : on part de A et on visite B, C et E dans l'ordre optimal trouvé par l'heuristique. On affiche le tour, le coût total et le détail par tronçon.

In [ ]:
tour = solve_tsp(["B", "C", "E"], start="A", end=None, costs=pair_costs)

tour_cost = sum(
    min(pair_costs.get(f"{tour[i]}|{tour[i+1]}", math.inf),
        pair_costs.get(f"{tour[i+1]}|{tour[i]}", math.inf))
    for i in range(len(tour) - 1)
)

print("Tour optimal (plus proche voisin + 2-opt) :")
print(" -> ".join(tour))
print(f"Temps de trajet total : {tour_cost:.2f} min")

print("\nDétail par tronçon :")
for i in range(len(tour) - 1):
    frm, to = tour[i], tour[i+1]
    leg_cost = pair_costs.get(f"{frm}|{to}", pair_costs.get(f"{to}|{frm}", math.inf))
    path_nodes = pair_paths.get(f"{frm}|{to}", {}).get("nodes", [frm, to])
    print(f"  {frm} -> {to}  chemin : {' -> '.join(path_nodes)}  ({leg_cost:.2f} min)")

**Cellule 4** — Vérification par force brute : on teste toutes les permutations possibles (faisable ici car seulement 3 villes à ordonner) et on compare avec le résultat de l'heuristique.

In [ ]:
from itertools import permutations

def perm_cost(perm, start, costs):
    route = ([start] if start else []) + list(perm)
    return sum(
        min(costs.get(f"{route[i]}|{route[i+1]}", math.inf),
            costs.get(f"{route[i+1]}|{route[i]}", math.inf))
        for i in range(len(route) - 1)
    )

best_perm, best_cost = None, math.inf
for perm in permutations(["B", "C", "E"]):
    c = perm_cost(perm, "A", pair_costs)
    if c < best_cost:
        best_cost = c
        best_perm = perm

optimal_route = ["A"] + list(best_perm)
print("Optimal (force brute) :", " -> ".join(optimal_route), f"({best_cost:.2f} min)")
print("Heuristique           :", " -> ".join(tour),           f"({tour_cost:.2f} min)")
gap = (tour_cost - best_cost) / best_cost * 100 if best_cost > 0 else 0
print(f"Écart d'optimalité : {gap:.1f}%")

---

## 8. Récapitulatif des Formules

| Métrique | Formule | Unité |
|----------|---------|-------|
| Efficacité trafic | $\text{eff} = \max(0{,}05,\ 1 - 0{,}6t)$ | — |
| Vitesse effective | $v^{\text{eff}} = \min(v, v^{\text{max}}) \cdot \text{eff}$ | km/h |
| Temps par segment | $T = \dfrac{d}{v^{\text{eff}}} \times 60$ | min |
| Carburant par segment | $F = d \cdot (0{,}07 + 0{,}05t)$ | L |
| Temps total d'une route | $T_{\text{route}} = \displaystyle\sum_{s} T_s$ | min |
| Distance totale d'une route | $D_{\text{route}} = \displaystyle\sum_{s} d_s$ | km |
| Carburant total d'une route | $F_{\text{route}} = \displaystyle\sum_{s} F_s$ | L |
| Score flotte (temps) | $S = \max_v T_v^{\text{route}}$ | min |
| Score flotte (distance/carburant) | $S = \displaystyle\sum_v X_v^{\text{route}}$ | km ou L |

> Le **score temps** prend le maximum sur tous les véhicules (le plus lent détermine la durée totale).  
> Les **scores distance et carburant** sont la somme sur tous les véhicules.

---

## 9. Vérification des Contraintes

Après le calcul de la route, chaque véhicule est contrôlé par rapport aux contraintes optionnelles définies dans l'objectif :

```python
if obj.maxTime     is not None and temps_route     > obj.maxTime:        violation
if obj.maxDistance is not None and distance_route  > obj.maxDistance:    violation
if obj.totalUnits  > 0         and unités_chargées > véhicule.capacité:  violation
```

Une route est marquée **faisable** si et seulement si elle ne présente aucune violation.  
Le résultat global est faisable uniquement si toutes les routes de véhicules sont faisables.